In [ ]:
!git clone https://github.com/sumit760/Codeglue.git

In [ ]:
%cd /kaggle/working
!ls

In [ ]:
!git config user.name "PrudhviGowroju"
!git config user.email "Prudhvi90hrithik@gmail.com"
!git config --list | grep user

In [ ]:
!git status


In [ ]:
!git checkout Prudhvi

In [ ]:
!git branch

In [ ]:
# 1. Install/upgrade necessary libraries
!pip install -q transformers datasets wandb evaluate scikit-learn

from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login
from datasets import load_dataset

# 2. Authenticate using Kaggle Secrets
try:
    secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    login(token=secrets.get_secret('HF_TOKEN'))
    wandb.login()
    print("✅ Successfully logged into Hugging Face and Weights & Biases!")
except Exception as e:
    print(f"⚠️ Secret loading error: {e}")

# 3. Load the dataset
print("\nDownloading the CodeXGLUE dataset...")
dataset = load_dataset("google/code_x_glue_cc_defect_detection")

# 4. Inspect the data structure
print("\n--- DATASET STRUCTURE ---")
print(dataset)

print("\n--- SAMPLE ITEM (Row 0) ---")
print(dataset['train'][0])

In [ ]:
import json
from transformers import AutoTokenizer

# 1. Load the tokenizer for CodeBERTa
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the preprocessing function
def preprocess_function(examples):
    # Clean the code: replace multiple spaces/newlines with a single space
    cleaned_code = [" ".join(code.split()) for code in examples["func"]]
    
    # Tokenize the code (truncating to 512 tokens to fit BERT's maximum length)
    model_inputs = tokenizer(
        cleaned_code,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    
    # Convert Boolean target to integers (0 = Safe, 1 = Defective)
    model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
    
    return model_inputs

# 3. Apply the processing to all splits and drop old columns
print("Cleaning, tokenizing, and dropping unused columns. This may take a minute...")
columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
tokenized_dataset = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=columns_to_remove
)

# 4. Create and save the id2label mapping file
id2label = {0: "Safe", 1: "Defective"}

with open("id2label.json", "w") as f:
    json.dump(id2label, f)

print("✅ Data preparation complete! id2label.json is saved locally.")
print("\n--- TOKENIZED DATASET STRUCTURE ---")
print(tokenized_dataset)

In [ ]:
import wandb
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Load the Model
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading {model_name} for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Safe", 1: "Defective"},
    label2id={"Safe": 0, "Defective": 1}
)

# 2. Define Metrics (Accuracy and F1)
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

# 3. Initialize W&B for Version 1
wandb.init(
    project="mlops-assignment3", 
    name="run-v1", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "version": "v1",
        "platform": "Kaggle"
    }
)

# 4. Set Training Arguments
training_args = TrainingArguments(
    output_dir='./results_v1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v1",
    fp16=True # Accelerates training on Kaggle GPUs
)

# 5. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v1...")
trainer.train()

# 6. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset['test'])
print(test_results)

wandb.finish()

In [ ]:
# 1. Initialize W&B for Version 2
wandb.init(
    project="mlops-assignment3", 
    name="run-v2", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 5e-5, # HYPERPARAMETER CHANGE
        "version": "v2",
        "platform": "Kaggle"
    }
)

# 2. Set Training Arguments for v2
training_args_v2 = TrainingArguments(
    output_dir='./results_v2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5, # HYPERPARAMETER CHANGE
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v2",
    fp16=True 
)

# 3. Initialize Trainer and Train
trainer_v2 = Trainer(
    model=model,
    args=training_args_v2,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v2...")
trainer_v2.train()

# 4. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set for run v2...")
test_results_v2 = trainer_v2.evaluate(tokenized_dataset['test'])
print(test_results_v2)

wandb.finish()